In [2]:
import pandas as pd
from datetime import datetime
import sqlite3
import random
from sqlalchemy import create_engine, text
import os

In [3]:
# conn = sqlite3.connect("./my_database.db")

In [4]:
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "vendor_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
conn = create_engine(DATABASE_URL)

In [5]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [6]:
vendor_ids

,vendor_id
0,1d7e9e8f-40be-49dc-9672-3aa65ee2f0d0
1,f1c4b27c-6e22-48a1-a23c-8d646460dcc9
2,87557121-007b-4e66-9a6e-733b6e2bc363
3,600ad80b-c837-4db6-b132-5c5f8b995ef6
4,9deb59e9-2b41-4fa0-b272-fa2b3dc749de
5,4c23bea9-bb9e-4064-83d9-a54f94a01185
6,3dd270ee-41ef-47ed-98c8-22b2af21b4aa
7,ad31d7e7-9a34-4eb1-8030-317d6d9d1a4d
8,62aab707-f2c5-4283-86fb-7bdb83f4ac9b
9,0709cc65-6c9f-4f4d-8ae3-f0bbc2a2a1a8


In [7]:
test_vendor = vendor_ids.iloc[1]['vendor_id']

In [8]:
test_vendor

'f1c4b27c-6e22-48a1-a23c-8d646460dcc9'

In [9]:
def active_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and status = 'IN_PROGRESS'
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [10]:
active_tickets(test_vendor)

{'vendor_id': 'f1c4b27c-6e22-48a1-a23c-8d646460dcc9',
 'active tickets': np.int64(110)}

In [11]:
def vendor_on_time_delivery(vendor_id):
    df = pd.read_sql(
        f"""
        select * 
        from ticket
        where vendor_id = '{vendor_id}'
        """,
        con=conn
    )
    df['on_time'] = (
        df['completed_at'].notna() &
        (df['completed_at'] <= df['due_date'])
    )

    df = df.groupby('vendor_id', as_index=False).agg(on_time_count=("on_time", "sum"),ticket_count=("ticket_id", "count"))
    df['on_time_pct'] = df['on_time_count'] / df['ticket_count'] * 100
    print(df[['vendor_id', 'on_time_count', 'ticket_count', 'on_time_pct']])
    on_time_pct = df['on_time_pct'].iloc[0].round(2)
    return {'vendor_id': vendor_id, 'on_time_pct': on_time_pct}

In [12]:
vendor_on_time_delivery(test_vendor)

                              vendor_id  on_time_count  ticket_count  \
0  f1c4b27c-6e22-48a1-a23c-8d646460dcc9             96           439   

   on_time_pct  
0    21.867882  


{'vendor_id': 'f1c4b27c-6e22-48a1-a23c-8d646460dcc9',
 'on_time_pct': np.float64(21.87)}

In [13]:
def flagged_tickets(vendor_id):
    df = pd.read_sql(
        f"""
            select count(*) as flagged_tickets from ticket
            where vendor_id='{vendor_id}' and anomaly_flag=True
        """, con=conn
    )
    flagged_ticket_count = df['flagged_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'flagged_ticket_count': flagged_ticket_count}

In [14]:
flagged_tickets(test_vendor)

{'vendor_id': 'f1c4b27c-6e22-48a1-a23c-8d646460dcc9',
 'flagged_ticket_count': np.int64(227)}

## Completed Today (work in progress due to logic of sample data)

In [15]:
df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{test_vendor}' and status='COMPLETED'
        """, parse_dates=['assigned_at', 'completed_at', 'created_at', 'updated_at', 'start_time', 'due_date'], con=conn
    )

In [16]:
max(df['completed_at'])

Timestamp('2026-03-31 19:56:24.491489+0000', tz='UTC')

In [17]:
def completed_tickets_today(vendor_id):
    df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{vendor_id}'
        """, con=conn
    )
    return df

In [18]:
# completed_tickets_today(test_vendor)

## Average Duration

In [19]:
def average_duration(vendor_id):
    df = pd.read_sql(
        f"""
        select * from ticket
        where vendor_id = '{vendor_id}' and status='COMPLETED'
        """,
        con=conn,
        parse_dates=['completed_at', 'start_time']
    )
    df['duration'] = df['completed_at'] - df['start_time']
    avg_duration = df['duration'].mean()
    return {'vendor_id': vendor_id, 'avg_duration': avg_duration}

In [20]:
average_duration(test_vendor)

{'vendor_id': 'f1c4b27c-6e22-48a1-a23c-8d646460dcc9',
 'avg_duration': Timedelta('2 days 04:49:25.194841')}